In [1]:
%reload_ext autoreload
%autoreload 2
%matplotlib inline

In [2]:
import torch
from nanodrz.model import DiarizeGPT, Config
from nanodrz import data
from nanodrz.data import artificial_diarisation_sample
from nanodrz.utils import visualise_annotation, play
from nanodrz.download import dl_scp_file
from nanodrz import format_conversions as format

device = torch.device("cuda:1")
torch.cuda.set_device(device)
ckpt = torch.load("/home/harry/runs/nanodrz/1706953646/0030000.pt", map_location=device)

RuntimeError: CUDA error: invalid device ordinal
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1.
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:
config = Config(**ckpt["config"])
model:DiarizeGPT = DiarizeGPT.from_pretrained(ckpt).cuda()

In [ ]:
# Use the same parameters that the model was trained on to generate a sample
audio, labels = artificial_diarisation_sample(data.librilight_small(), min_secs=120, num_speakers=5, max_secs=120)
reference = format.labels_to_annotation(labels)

nlabels = model.generate(audio.cuda(), temperature=1, max_steps=(len(labels))*3, top_p=.5)
print("\n".join([str(n) for n in nlabels]))

for l in nlabels:
    l[2] = l[2]+ "'"

hypothesis = format.labels_to_annotation(nlabels)
visualise_annotation(labels+nlabels)

from pyannote.core import Annotation
from pyannote.metrics.diarization import DiarizationErrorRate

metric = DiarizationErrorRate()
metric(reference, hypothesis)